# Projet — Base de données simple VS Ontologie

Mêmes données (un organigramme d'entreprise), deux approches côte à côte. On va voir **exactement** où la simple table SQL coince, et comment l'ontologie répond.

Deux scénarios concrets :
1. **Hiérarchie de types** : « donne-moi tous les employés » alors que chacun a un rôle précis (Manager, Ingénieur...).
2. **Relation transitive** : « qui se trouve au-dessus de David dans toute la chaîne ? ».

> Prérequis : `pip install -r requirements.txt`. Exécute les cellules dans l'ordre.

## Étape A — Les données dans une table SQL

On charge le CSV dans une table SQLite (inclus dans Python). C'est notre « base de données simple ».

In [1]:
import csv
import sqlite3

connexion = sqlite3.connect(":memory:")
connexion.execute("CREATE TABLE employe (id TEXT, nom TEXT, role TEXT, superieur_id TEXT)")

with open("entreprise.csv", encoding="utf-8") as fichier:
    lignes = [
        (l["id"], l["nom"], l["role"], l["superieur_id"] or None)
        for l in csv.DictReader(fichier)
    ]
connexion.executemany("INSERT INTO employe VALUES (?, ?, ?, ?)", lignes)

# Affichage de la table
for ligne in connexion.execute("SELECT * FROM employe"):
    print(ligne)

('e1', 'Alice', 'Directrice', None)
('e2', 'Bob', 'Manager', 'e1')
('e3', 'Carol', 'Manager', 'e1')
('e4', 'David', 'Ingenieur', 'e2')
('e5', 'Emma', 'Ingenieur', 'e2')
('e6', 'Frank', 'Ingenieur', 'e3')


## Scénario 1 (SQL) — « Tous les employés ? »

Le rôle est juste du **texte** dans une colonne. La table ne sait pas qu'un Manager *est* un Employé. Une recherche directe `role = 'Employe'` ne renvoie donc... rien.

In [2]:
vide = connexion.execute("SELECT nom FROM employe WHERE role = 'Employe'").fetchall()
print("WHERE role = 'Employe' :", vide, "-> rien !")

# Il faut ENUMERER tous les roles a la main (et y penser si on en ajoute un nouveau)
roles = ("Directrice", "Manager", "Ingenieur")
q = f"SELECT nom FROM employe WHERE role IN ({','.join('?' for _ in roles)})"
print("En listant les roles a la main :", [r[0] for r in connexion.execute(q, roles)])

WHERE role = 'Employe' : [] -> rien !
En listant les roles a la main : ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']


## Scénario 2 (SQL) — « Toute la chaîne au-dessus de David ? »

Le supérieur **direct** est facile (une colonne). Mais remonter **toute** la chaîne exige une requête **récursive** (`WITH RECURSIVE`) : ça marche, mais c'est du code à écrire et à maintenir soi-même.

In [3]:
requete_recursive = """
WITH RECURSIVE chaine(id, superieur_id) AS (
    SELECT id, superieur_id FROM employe WHERE id = 'e4'
    UNION ALL
    SELECT e.id, e.superieur_id
    FROM employe e
    JOIN chaine c ON e.id = c.superieur_id
)
SELECT sup.nom
FROM chaine
JOIN employe sup ON sup.id = chaine.superieur_id
"""
chaine = [r[0] for r in connexion.execute(requete_recursive)]
print("Chaine au-dessus de David (SQL recursif) :", chaine)

Chaine au-dessus de David (SQL recursif) : ['Alice', 'Bob']


## Étape B — Les mêmes données en ontologie

On charge `entreprise.ttl` et on lance le **raisonneur OWL**. Le schéma (sous-classes, propriété transitive, propriété inverse) est déclaré **une seule fois** ; le raisonneur en déduit le reste.

In [4]:
from rdflib import Graph
from owlrl import DeductiveClosure, OWLRL_Semantics

graphe = Graph()
graphe.parse("entreprise.ttl", format="turtle")
print(f"Triplets avant raisonnement : {len(graphe)}")

DeductiveClosure(OWLRL_Semantics).expand(graphe)
print(f"Triplets apres raisonnement : {len(graphe)}")


def afficher(requete: str) -> None:
    for ligne in graphe.query(requete):
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)

Triplets avant raisonnement : 35

Triplets apres raisonnement : 218


## Scénario 1 (ontologie) — « Tous les employés ? »

Cette fois, on demande simplement les `ex:Employe`. Grâce à `subClassOf`, le raisonneur a compris que Managers, Ingénieurs et Directrice **sont** des Employés. La requête est triviale et restera juste même si on ajoute un nouveau rôle plus tard.

In [5]:
afficher("""
PREFIX ex: <http://exemple.org/entreprise/>

SELECT ?nom WHERE {
    ?p a ex:Employe .
    ?p ex:nom ?nom .
}
""")

  {'nom': 'Bob'}
  {'nom': 'Frank'}
  {'nom': 'Carol'}
  {'nom': 'Emma'}
  {'nom': 'David'}
  {'nom': 'Alice'}


## Scénario 2 (ontologie) — « Toute la chaîne au-dessus de David ? »

`aPourSuperieur` est déclarée **transitive**. On a seulement écrit les liens directs (David→Bob, Bob→Alice), et le raisonneur en a déduit David→Alice. Pas de requête récursive : une requête plate suffit.

In [6]:
afficher("""
PREFIX ex: <http://exemple.org/entreprise/>

SELECT ?nomSuperieur WHERE {
    ex:david ex:aPourSuperieur ?sup .
    ?sup ex:nom ?nomSuperieur .
}
""")

  {'nomSuperieur': 'Bob'}
  {'nomSuperieur': 'Alice'}


## Bonus — Une relation gratuite : « qui Bob supervise-t-il ? »

On n'a **jamais** écrit la relation `supervise`. Mais comme elle est déclarée `owl:inverseOf ex:aPourSuperieur`, le raisonneur la déduit. On obtient les subordonnés (directs ET indirects, grâce à la transitivité combinée).

In [7]:
afficher("""
PREFIX ex: <http://exemple.org/entreprise/>

SELECT ?nomSubordonne WHERE {
    ex:bob ex:supervise ?sub .
    ?sub ex:nom ?nomSubordonne .
}
""")

  {'nomSubordonne': 'Emma'}
  {'nomSubordonne': 'David'}


## Bilan

| Question | Table SQL | Ontologie |
|---|---|---|
| Tous les employés | Énumérer les rôles à la main (fragile) | `?p a ex:Employe` (déduit via subClassOf) |
| Chaîne hiérarchique complète | Requête récursive à coder | Requête plate (propriété transitive) |
| Relation inverse (subordonnés) | À recalculer par une jointure | Déduite gratuitement (`owl:inverseOf`) |

**La table stocke les données ; l'ontologie stocke les données ET leur sens, ce qui permet de raisonner.**

Quand préférer une ontologie : données très reliées, règles métier riches, schéma évolutif, intégration de sources hétérogènes. Quand garder une simple base : données tabulaires régulières, gros volumes, transactions, besoin de performance brute. Voir le `README.md` de ce dossier pour le détail.